In [1]:
# Cell 1: 导入依赖 & 配置参数
import os

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import re
import psycopg2
from pathlib import Path
from sentence_transformers import SentenceTransformer

# ---------- 数据库连接配置 ----------
DB_CONFIG = {
    "dbname": "Law_app",
    "user": "my_pgsql",
    "password": "123123",
    "host": "localhost",
    "port": 5433,
}

# ---------- 嵌入模型 ----------
MODEL_NAME = "BAAI/bge-large-zh-v1.5"
model = SentenceTransformer(MODEL_NAME)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [3]:
DB_CONFIG = {
    "dbname": "Law_app",
    "user": "my_pgsql",
    "password": "123123",
    "host": "localhost",
    "port": 5433,
}
# 测试数据库连接
try:
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    cur.execute("SELECT version();")
    version = cur.fetchone()
    print(f"数据库连接成功！PostgreSQL 版本: {version[0]}")
    cur.close()
    conn.close()
except psycopg2.OperationalError as e:
    print(f"数据库连接失败: {e}")
except Exception as e:
    print(f"未知错误: {e}")

数据库连接成功！PostgreSQL 版本: PostgreSQL 15.4 (Debian 15.4-2.pgdg120+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 12.2.0-14) 12.2.0, 64-bit


In [4]:
def parse_law_from_file(file_path: str):
    """
    从法律文本文件中解析法条。
    Args:
        file_path: 法律文件路径
    Returns:
        law_title: 法律名称
        articles: list[dict] 每个元素包含 chapter, article_number, content
    """
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    law_title = Path(file_path).stem

    # ---------- 匹配章/节标题 ----------
    # 支持三种格式: "第X章 XXX"、"X、XXX" 或无章节
    chapter_pattern = re.compile(
        r"^(?:第([一二三四五六七八九十百千零]+)章\s*(.*))"
        r"|^(?:([一二三四五六七八九十百千零]+)、(.+))",
        re.MULTILINE,
    )
    chapter_matches = list(chapter_pattern.finditer(text))

    if not chapter_matches:
        sections = [("", 0, len(text))]
    else:
        sections = []
        for i, m in enumerate(chapter_matches):
            if m.group(1):  # "第X章 XXX" 格式
                chapter_name = f"第{m.group(1)}章 {m.group(2).strip()}"
            else:  # "X、XXX" 格式
                chapter_name = f"{m.group(3)}、{m.group(4).strip()}"
            start = m.start()
            end = chapter_matches[i + 1].start() if i + 1 < len(chapter_matches) else len(text)
            sections.append((chapter_name, start, end))

    # ---------- 匹配条文起始位置 ----------
    article_start_re = re.compile(r"第([一二三四五六七八九十百千零]+)条\s*")

    # ---------- 过滤施行日期条款 ----------
    _DATE_CLAUSE_RE = re.compile(
        r"(?:本条例|本规定|本法|本解释)自\d{4}年\d{1,2}月\d{1,2}日起施行"
    )

    def is_effective_date_clause(content: str) -> bool:
        c = content.strip()
        return len(c) < 80 and bool(_DATE_CLAUSE_RE.search(c))

    articles = []
    for section_name, sec_start, sec_end in sections:
        section_text = text[sec_start:sec_end]
        # 找到该章节内所有"第X条"的位置
        article_starts = list(article_start_re.finditer(section_text))

        for i, m in enumerate(article_starts):
            article_num = m.group(1)
            content_start = m.end()  # "第X条"之后
            # 内容区间: 当前条文起始 到 下一条文起始(或章节末尾)
            if i + 1 < len(article_starts):
                content_end = article_starts[i + 1].start()
            else:
                content_end = len(section_text)

            raw_content = section_text[content_start:content_end].strip()

            # 以中文句号作为法条内容的自然结束边界
            last_period = raw_content.rfind("。")
            if last_period != -1:
                raw_content = raw_content[:last_period + 1]

            if not raw_content or is_effective_date_clause(raw_content):
                continue

            articles.append(
                {
                    "chapter": section_name or "",
                    "article_number": f"第{article_num}条",
                    "content": raw_content,
                }
            )

    return law_title, articles

In [10]:
# Cell 3: 数据库建表（首次运行执行一次, 表已存在则跳过）
def create_table_if_not_exists():
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
    cur.execute("""
        CREATE TABLE IF NOT EXISTS marriage_law (
            id SERIAL PRIMARY KEY,
            law_title TEXT NOT NULL,
            chapter TEXT,
            article_number TEXT NOT NULL,
            content TEXT NOT NULL,
            embedding VECTOR(1024),
            UNIQUE(law_title, article_number)
        );
        -- 索引按需创建, 见 create-schema-template.sql
    """)
    conn.commit()
    cur.close()
    conn.close()
    print("表已就绪。")


create_table_if_not_exists()

表已就绪。


In [12]:
# 创建 agent_memory 表 (长期记忆 — PostgreSQL + pgvector)
def create_memory_table():
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
    cur.execute("""
        CREATE TABLE IF NOT EXISTS agent_memory (
            id          SERIAL PRIMARY KEY,
            thread_id   TEXT NOT NULL DEFAULT 'default',
            memory_type TEXT NOT NULL DEFAULT 'general',
            content     TEXT NOT NULL,
            embedding   VECTOR(1024),
            metadata    JSONB DEFAULT '{}'::jsonb,
            created_at  TIMESTAMP DEFAULT NOW()
        );
    """)
    conn.commit()
    cur.close()
    conn.close()
    print("agent_memory 表已就绪。")

create_memory_table()

agent_memory 表已就绪。


In [5]:
# Cell 4: 向量化并插入数据库
def insert_articles(law_title: str, articles: list[dict]):
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    sql = """
        INSERT INTO marriage_law (law_title, chapter, article_number, content, embedding)
        VALUES (%s, %s, %s, %s, %s)
        ON CONFLICT (law_title, article_number) DO UPDATE
        SET content = EXCLUDED.content,
            embedding = EXCLUDED.embedding,
            chapter = EXCLUDED.chapter
    """
    for art in articles:
        embedding = model.encode(art["content"], normalize_embeddings=True).tolist()
        cur.execute(
            sql,
            (
                law_title,
                art.get("chapter", ""),
                art["article_number"],
                art["content"],
                embedding,
            ),
        )
    conn.commit()
    cur.close()
    conn.close()
    print(f"成功插入/更新 {len(articles)} 条记录。")

In [6]:
def insert_law_vector(law_title: str, articles: list[dict]):
    """
    将解析后的法条数据插入 law_vector 表。
    Args:
        law_title: 法律名称
        articles: parse_law_from_file 返回的法条列表
    """
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()

    sql = """
        INSERT INTO law_vector (law_title, chapter, article_number, content, embedding)
        SELECT %s, %s, %s, %s, %s
        WHERE NOT EXISTS (
            SELECT 1 FROM law_vector
            WHERE law_title = %s AND article_number = %s
        )
    """
    inserted = 0
    skipped = 0

    for art in articles:
        embedding = model.encode(art["content"], normalize_embeddings=True).tolist()
        cur.execute(
            sql,
            (
                law_title,
                art.get("chapter", ""),
                art["article_number"],
                art["content"],
                embedding,
                law_title,
                art["article_number"],
            ),
        )
        if cur.rowcount > 0:
            inserted += 1
        else:
            skipped += 1

    conn.commit()
    cur.close()
    conn.close()
    print(f"law_vector: 成功插入 {inserted} 条, 跳过 {skipped} 条(已存在)。")

In [7]:
from pathlib import Path

law_dir = Path(r"E:\\LangChain\\lawApp_LangGraph\\Documents\\LawDocument")
txt_files = sorted(law_dir.glob("*.txt"))

if not txt_files:
    print(f"目录 {law_dir} 下未找到 .txt 文件。")
else:
    print(f"共发现 {len(txt_files)} 个法律文件:\n")
    for fp in txt_files:
        print(f"  - {fp.name}")

    # for fp in txt_files:
    #     print(f"\n处理: {fp.name}")
    #     law_title, articles = parse_law_from_file(str(fp))
    #     print(f"  解析到 {len(articles)} 条")
    #     insert_law_vector(law_title, articles)

共发现 3 个法律文件:

  - 中华人民共和国反家庭暴力法.txt
  - 中华人民共和国妇女权益保障法.txt
  - 最高人民法院关于审理涉彩礼纠纷案件.txt


In [8]:
def parse_law_from_file_Version2(file_path: str):
    """
    解析法律文本文件。
    逻辑:
        1. 按章节拆分全文 (第X章 / X、格式 / 无章节则整篇)
        2. 每章内: 以行首"第XX条"为起点, 以中文句号"。"为终点提取条文
        3. 条文内部的"第XX条"引用不参与切分
    Args:
        file_path: 法律文件路径
    Returns:
        law_title: 法律名称
        articles: list[dict]
    """
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    law_title = Path(file_path).stem

    # ── 阶段一: 拆分章节 ──────────────────────────────
    marks = []  # [(pos, name)]

    # 格式A — "第X章 XXX"  (允许前导空格, 如"   第一章 基本 规定")
    for m in re.finditer(
        r"^\s*第([一二三四五六七八九十百千零]+)章\s*(.+)", text, re.MULTILINE
    ):
        marks.append((m.start(), f"第{m[1]}章 {m[2].strip()}"))

    # 格式B — "X、XXX"  (独立短行, ≤40字, 避免匹配文内子项)
    # for m in re.finditer(
    #         r"^\s*([一二三四五六七八九十百千零]+)、(.{1,40})$",
    #         text, re.MULTILINE):
    #     if not any(abs(m.start() - p) < 2 for p, _ in marks):
    #         marks.append((m.start(), f"{m[1]}、{m[2].strip()}"))

    marks.sort(key=lambda x: x[0])

    sections = []
    if not marks:
        sections = [("", 0, len(text))]
    else:
        for i, (start, name) in enumerate(marks):
            end = marks[i + 1][0] if i + 1 < len(marks) else len(text)
            sections.append((name, start, end))

    # ── 阶段二: 逐章提取条文 ────────────────────────────
    # 条文起点: 行首 "第XX条"  (同样允许前导空格)
    HEAD = re.compile(r"^\s*第([一二三四五六七八九十百千零]+)条\s*", re.MULTILINE)

    # 施行日期子句 (不视为法条)
    DATE_CLAUSE = re.compile(
        r"(?:本条例|本规定|本法|本解释)自\d{4}年\d{1,2}月\d{1,2}日起施行"
    )

    def _skip(content: str) -> bool:
        c = content.strip()
        return not c or (len(c) < 80 and DATE_CLAUSE.search(c))

    articles = []
    for idx, (sec_name, sec_start, sec_end) in enumerate(sections, 1):
        zone = text[sec_start:sec_end]
        heads = list(HEAD.finditer(zone))

        for i, h in enumerate(heads):
            num = h[1]
            body_start = h.end()
            body_end = heads[i + 1].start() if i + 1 < len(heads) else len(zone)
            body = zone[body_start:body_end].strip()

            cut = body.rfind("。")
            if cut != -1:
                body = body[: cut + 1]

            if _skip(body):
                continue

            articles.append(
                {
                    "chapter": sec_name or "",
                    "article_number": f"第{num}条",
                    "content": body,
                }
            )

        print(
            f"\r  解析进度: {idx}/{len(sections)} 章节, 已收集 {len(articles)} 条",
            end="",
        )

    print()
    return law_title, articles

In [ ]:
# 批量解析并入库: Error 目录下所有法律文件
from pathlib import Path

law_dir = Path(r"E:\LangChain\lawApp_LangGraph\Documents\LawDocument")
txt_files = sorted(law_dir.glob("*.txt"))

print(f"目录: {law_dir}")
print(f"共发现 {len(txt_files)} 个文件:\n")
for fp in txt_files:
    print(f"  - {fp.name}")

print()

for fp in txt_files:
    print(f"\n{'='*50}")
    print(f"文件: {fp.name}")

    law_title, articles = parse_law_from_file_Version2(str(fp))
    total = len(articles)

    if not articles:
        print("  未解析到任何法条, 跳过")
        continue

    sql = """
        INSERT INTO law_vector (law_title, chapter, article_number, content, embedding)
        VALUES (%s, %s, %s, %s, %s)
    """
    for i, art in enumerate(articles, 1):
        embedding = model.encode(art["content"], normalize_embeddings=True).tolist()
        cur.execute(sql, (
            law_title,
            art.get("chapter", ""),
            art["article_number"],
            art["content"],
            embedding,
        ))
        if i % 50 == 0 or i == total:
            conn.commit()
            print(f"\r  入库: {i}/{total}", end="")

    conn.commit()
    cur.close()
    conn.close()
    print(f"\n  完成: 清除 {deleted} 条旧记录, 插入 {total} 条")


目录: E:\LangChain\lawApp_LangGraph\Documents\LawDocument\Error
共发现 3 个文件:

  - 婚姻登记条例.txt
  - 最高人民法院关于适用《中华人民共和国民法典》婚姻家庭编的解释（一）.txt
  - 最高人民法院关于适用《中华人民共和国民法典》婚姻家庭编的解释（二）.txt


文件: 婚姻登记条例.txt
  解析进度: 6/6 章节, 已收集 27 条
  入库: 27/27
  完成: 清除 0 条旧记录, 插入 27 条

文件: 最高人民法院关于适用《中华人民共和国民法典》婚姻家庭编的解释（一）.txt
  解析进度: 1/1 章节, 已收集 90 条
  入库: 90/90
  完成: 清除 0 条旧记录, 插入 90 条

文件: 最高人民法院关于适用《中华人民共和国民法典》婚姻家庭编的解释（二）.txt
  解析进度: 1/1 章节, 已收集 22 条
  入库: 22/22
  完成: 清除 0 条旧记录, 插入 22 条


In [2]:
import torch

print(torch.__version__)  # 不报错就说明 OK


OSError: [WinError 127] 找不到指定的程序。 Error loading "f:\Anaconda_env\lawApp_langGraph\Lib\site-packages\torch\lib\shm.dll" or one of its dependencies.

In [ ]:
{
    "query": "老婆在外出轨,我要求离婚,她死活不答应,怎么办我要离婚",
    "session_id": "string",
    "final_answer": "收到你的问题。我能理解你此刻的愤怒和想要尽快结束这段关系的迫切心情。请先冷静下来，我们来把这件事在法律上拆解清楚，找到最有效的解决办法。\n\n**一、核心法律定性**\n\n你目前面临的是一个典型的**夫妻感情破裂离婚纠纷**。关键点不在于她“是否出轨”（这属于道德和情感问题，法律上只关注其导致的法律后果），而在于 **“她不同意离婚”** 。这是你当前最直接的法律障碍。\n\n**二、法院会怎么判？（参考案例思路）**\n\n根据中国《民法典》和相关司法实践，法院判决离婚的唯一标准是 **“夫妻感情确已破裂”** 。如果一方坚决不同意，法院不会仅仅因为“出轨”就立刻判决离婚。法院的裁判思路通常是：\n\n1.  **第一次起诉，大概率不判离。** 对于第一次起诉离婚，尤其在被告坚决不同意的情况下，法院倾向于给双方一个“冷静期”或“修复期”。只要被告能拿出“没有证据证明感情彻底破裂”的理由（比如承认错误、愿意改正、强调家庭孩子），法院大概率会判决**不准予离婚**。这指的是你第一次起诉，如果她没有过错方导致的重大法定义务违反（如家暴、遗弃、赌博等），第一次基本很难离掉。\n2.  **关键证据的作用：** 如果你能提供充分、合法的证据证明她的出轨行为已经导致夫妻感情“确已破裂”（例如，她长期与第三者同居、重婚、或因出轨导致家庭无法维持等），法院在第二次或后续起诉时，判决离婚的可能性会显著增加。但“普通出轨”不等于“感情必然破裂”，法院会综合考量。\n\n**三、你现在最需要关注的三大风险**\n\n1.  **证据陷阱：** 如果你现在冲动地去跟踪、偷拍、暴力取证，或者使用非法手段获取的证据（如侵入他人设备、窃听、安装针孔摄像头等），这些证据不仅法院不会采纳，甚至可能让你自己陷入侵权、行政或刑事风险（如侵犯隐私、非法侵入住宅）。\n2.  **拖延战术：** 她“死活不答应”这个状态，意味着她很可能在利用法律程序拖延时间。拖延的时间越长，对你越不利：财产可能被转移、抚养权争夺可能更复杂、你的情绪和精力消耗越大。\n3.  **情绪操作：** 她可能用“为了孩子”、“我改过自新，你再给我一次机会”等话术来试图动摇你。记住，**法院只认法律事实和证据**。你的情绪越稳定，越能做出理智决策。\n\n**四、给你的具体可行建议（下一步做什么）**\n\n不要和她打嘴仗，立刻进入**诉讼流程**。你唯一的、最有效的出路就是**向法院提起离婚诉讼**。\n\n**第一步：冷静收集证据（合法、有效）**\n*   **核心证据：** 证明她“与他人同居”或“与第三者保持稳定、长期关系”的证据。比如：\n    *   她与第三者的微信聊天记录、短信、通话记录（**注意：要合法获取！** 比如她放在桌上的手机，你作为配偶有权查看，但不能强行解锁、破解密码、或者安装监控软件）。\n    *   她与第三者的亲密照片、视频（**务必在公开场合拍摄，如公共场所、酒店大堂等，不能是在酒店房间内偷拍**）。\n    *   她承认出轨的录音、道歉信、悔过书等（**录音需要是双方在场，不能是单方偷录且内容不涉及侵害她人合法权益的对话**）。\n    *   能证实她与第三者以夫妻名义共同生活的物证（如共同租房合同、水电费账单等）。\n*   **辅助证据：** 证明你们夫妻感情本就不好、你为此感到痛苦、已经无法共同生活、分床分居时间的记录等。\n\n**第二步：立即咨询专业离婚律师**\n*   **为什么必须找律师？** 离婚官司非常专业，涉及证据规则、诉讼策略、财产分割、孩子抚养权等复杂问题。你自己去打，很容易在程序上和证据上吃亏。花几千块钱请个靠谱的律师，比你自己瞎折腾一年更划算。\n*   **找谁？** 找擅长处理“家事纠纷”或“婚姻家事”的律师，不要找刑事辩护或劳动争议律师。\n\n**第三步：正式向法院起诉**\n*   **起诉状怎么写？** 核心诉求是“请求法院判决解除婚姻关系”，并附上事实理由（出轨导致感情破裂）和你的诉讼请求（孩子、财产等）。\n*   **诉讼请求要清晰：** 要求分割夫妻共同财产、要求她承担出轨的过错赔偿责任（法律上叫“离婚损害赔偿”）、要求孩子抚养权归你（如果孩子是你的且你主张抚养权）。\n*   **做好打“持久战”的准备：** 第一次诉讼（从立案到一审判决）通常需要3-6个月。如果第一次判不离，你需要在**判决生效后过6个月**才能再次起诉。所以，最快的情况下，你也要10-12个月才能拿到离婚判决。\n\n**第四步：应对她的“不同意”**\n*   她不同意，没关系。你只需要在法庭上向法官清晰、理性地陈述感情破裂的事实和证据，表达离婚的坚定决心。你的律师会帮你处理。\n*   **不要被她情绪绑架。** 她的选择是她的事，你的选择是你的事。你唯一要做的，就是按照法律程序走。\n\n**总结一下最确定的结论：**\n\n*   **确定：** 只要她不同意，你**第一次起诉大概率离不掉**。\n*   **确定：** 你**必须立刻开始收集证据并找律师**，否则你白白浪费时间。\n*   **不确定：** 法院是否认定她的行为属于“感情破裂”的关键因素，这取决于你的证据是否够硬，以及法官的自由裁量。\n\n**现在，你最应该做的事：** 关掉手机，找一个安静的咖啡馆，列一份你现在能拿到的、证明她出轨的**客观证据清单**。然后，立刻去当地律所或通过朋友介绍，约见一位婚姻家事律师。不要再给她任何机会或浪费时间谈判了。你的时间、精力和法律上的主动权，才是最宝贵的。",
    "sources": [
        {
            "case_number": "",
            "year": "",
            "snippet": "[【被起诉离婚】在离婚案件中，如果被另一方起诉应该怎么办 ...] 这是最新的民法典，根据该条文可以看出，夫妻任何一方均有权要求离婚，向法院起诉，只要提交符合法定标准的夫妻感情破裂证据，即使对方不同意，法院依然有权强制 ... (http://www.lawyerlihun.com/htm/2024122/1434.htm)",
        },
        {
            "case_number": "",
            "year": "",
            "snippet": "[妻子出轨要离婚丈夫可以向第三者索要赔偿吗？] 离婚后是否可以追究第三者的责任，主要取决于具体情况。一般来说，第三者不是离婚案件的法律主体，而是离婚案件的诱因，因此不能直接起诉第三者。 (https://www.huichenglawyer.com/pufaku/65045.html)",
        },
        {
            "case_number": "",
            "year": "",
            "snippet": "[婚内出轨在离婚纠纷中的法律影响——上海女教师事件背后 ...] ... 另一方也无权要求损害赔偿金；. （2）若无过错方作为原告，必须在提起离婚诉讼的同时主张损害赔偿。若无过错作为被告，其既不同意离婚，也不提起损害 ... (https://www.kingandwood.com/cn/zh/insights/latest-thinking/the-legal-impact-of-marital",
        },
        {
            "case_number": "",
            "year": "",
            "snippet": "[在美国离婚一方不同意怎么办？] 只要一方提出离婚，法院一般会批准，即使另一方不同意。例如，如果一方表示两人已经分居半年以上且无法修复关系，法院通常不会拒绝批准。陈伟涛律师指出，这种 ... (https://www.faan.com/post/%E5%9C%A8%E7%BE%8E%E5%9B%BD%E7%A6%BB%E5%A9%9A%E4%B8%80%E6%96%B9%E4%B8%8D%E5%9",
        },
        {
            "case_number": "",
            "year": "",
            "snippet": "[一方要离婚一方不同意怎么处理] 如果一方坚决不肯离婚，法院一般不会判离，但是另一方6个月之后还可以再起诉，那么第二次法院一般会判决离婚。 2、如果一方不肯离婚怎么办？ 是不是要到法庭起诉？ 一方不同意 ... (https://zhuanlan.zhihu.com/p/344553949)",
        },
        {
            "case_number": "",
            "year": "",
            "snippet": "[有争议离婚] 1. 提交离婚诉讼. 如果夫妻双方无法协商解决离婚问题，一方需要向法院提交离婚诉讼，并通知对方。 · 2. 答辩与反诉. 被诉方（配偶）可以提交答辩书，说明不同意离婚或不同意对方 ... (https://www.lihun.law/%E6%9C%89%E4%BA%89%E8%AE%AE%E7%A6%BB%E5%A9%9A)",
        },
        {
            "case_number": "",
            "year": "",
            "snippet": "[写好离婚协议，体面说再见——如何科学设置抚养权相关条款] 协议离婚时，建议本着子女利益最大化原则，约定直接抚养子女的一方未经另一方同意，不得擅自改变子女的定居城市或国家。如果违约，违约一方承担违约责任，守约 ... (https://www.dehenglaw.com/CN/tansuocontent/0008/029754/7.aspx?AID=&BID=000000000000001984&",
        },
        {
            "case_number": "",
            "year": "",
            "snippet": "[男方不同意离婚，女方该怎么办？——上海离婚律师为你解答] 如果男方不同意离婚，女方可以通过法律途径解决，即向法院提起离婚诉讼。法院在审理离婚案件时，会首先进行调解，如果调解无效且感情确已破裂，法院将判决准予 ... (http://www.htclawfirm.com/hyjt/sslh/8325.html)",
        },
    ],
    "tool_calls": [
        "retrieve_legal_knowledge",
        "evaluate_case_relevance",
        "fetch_laws",
        "get_google_search",
        "analyze_legal_issue",
    ],
    "reasoning": [
        "用户咨询因配偶出轨且对方不同意离婚，如何实现离婚。涉及法律问题，需要检索相关案例和法条。",
        "按照流程：先检索案例，评估相关性，获取相关法条，并联网补充实务建议，最后综合分析提供可行路径。",
    ],
}
